# Task 08 -- Masked-HOOI convergence sensitivity (`docs/open_questions.md` §5.4)

Multi-initialization sensitivity check for masked HOOI's imputation of
land/ice cells, per `tucker_decomposition.md` §6's proposed mitigation.
See `tasks/08-masked-hooi-convergence.md`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2025, 12, 1, tzinfo=timezone.utc)

VARIABLE = "wsh"  # large contiguous land regions in the wave grid make this a good stress test
X_raw = load_training_window(DATA_DIR, T_START, T_END, [VARIABLE])
X = X_raw[VARIABLE].transpose("wave_lat", "wave_lon", "time").values

ocean_mask = ~np.isnan(X).any(axis=-1)
tucker_mask = np.broadcast_to(ocean_mask[:, :, None], X.shape).astype(np.float64)
X_filled = np.where(ocean_mask[:, :, None], X, 0.0)
print(f"{VARIABLE}: shape {X.shape}, {ocean_mask.mean():.1%} ocean")


## Multiple random initializations of masked HOOI, same rank/data

In [ ]:
R_LAT, R_LON, R_TIME = 10, 20, X_filled.shape[-1]
N_INITS = 8

reconstruction_errors = []
imputed_land_stats = []
for seed in range(N_INITS):
    core, factors = partial_tucker(
        X_filled, modes=[0, 1, 2], rank=[R_LAT, R_LON, R_TIME],
        init="random", tol=1e-5, n_iter_max=5000, random_state=seed,
        mask=tucker_mask,
    )
    recon = tl.tucker_to_tensor((core, factors))
    err = np.linalg.norm((recon - X_filled) * tucker_mask) / np.linalg.norm(X_filled * tucker_mask)
    reconstruction_errors.append(err)
    imputed_land_stats.append(recon[~ocean_mask].mean())
    print(f"init {seed}: relative error {err:.5f}, mean imputed-land value {imputed_land_stats[-1]:.4f}")

reconstruction_errors = np.array(reconstruction_errors)
imputed_land_stats = np.array(imputed_land_stats)
print(f"\nerror spread: mean {reconstruction_errors.mean():.5f}, std {reconstruction_errors.std():.5f} "
      f"({reconstruction_errors.std() / reconstruction_errors.mean():.1%} relative)")
print(f"imputed-land spread: mean {imputed_land_stats.mean():.4f}, std {imputed_land_stats.std():.4f}")


## Decision-rule diagnostic

Per `tucker_decomposition.md` §6: flag as "initialization-sensitive" if
either spread above has high relative std. Repeat for a variable with a
smaller contiguous land region as a control (e.g. `sigwh` at a coastal-
heavy sub-region) to see whether sensitivity correlates with land-region
size, as hypothesized. Record findings in `tasks/08-masked-hooi-convergence.md` §4.